# QUEST 4
_A/B-testing_

In [10]:
import pandas as pd
import sqlite3

We need to calculate what the delta between the first commit and the deadline was before
they visited the page for the first time and afterward. We need to do the same thing for
the control group too.

## 1) create a connection to the database using the library sqlite3

In [11]:
try:    
    connection = sqlite3.connect('../data/checking-logs.sqlite') 
    print("The connection is successful!")
except Exception:
    print("The connection is failed :(")

The connection is successful!


## 2) using only one query for each of the groups, create two dataframes: test_results and control_results with the columns time and avg_diff and only two rows

* time should have the values: after and before
* avg_diff contains the average delta among all the users for the time period
before each of them made their first visit to the page and afterward
* only take into account the users that have observations before and after

In [12]:
query = """
select time, avg(diff) AS avg_diff from (
	select uid, cast((julianDay(t.first_commit_ts) - julianDay(datetime(d.deadlines, 'unixepoch'))) * 24 as integer) as diff,
             case when t.first_commit_ts < t.first_view_ts then 'before' else 'after' 
             end as time
       from test t
       left join deadlines d on t.labname = d.labs
       where labname != 'project1'
      )
where uid in (select uid from(select uid,
                           	case when t.first_commit_ts < t.first_view_ts then 'before' else 'after' 
                           	end as time
                    			from test t
                    			left join deadlines d on t.labname=d.labs
                    			where labname != 'project1'
                    )
               group by uid
               having count(distinct time) = 2 -- only observations before and after
               )
group by time
"""
test_results = pd.io.sql.read_sql(query, connection)

То же самое для таблицы control:

In [13]:
query = """
select time, avg(diff) AS avg_diff from (
	select uid, cast((julianDay(c.first_commit_ts) - julianDay(datetime(d.deadlines, 'unixepoch'))) * 24 as integer) as diff,
             case when c.first_commit_ts < c.first_view_ts then 'before' else 'after' 
             end as time
       from control c
       left join deadlines d on c.labname = d.labs
       where labname != 'project1'
      )
where uid in (select uid from(select uid,
                           	case when c.first_commit_ts < c.first_view_ts then 'before' else 'after' 
                           	end as time
                    			from control c
                    			left join deadlines d on c.labname=d.labs
                    			where labname != 'project1'
                    )
               group by uid
               having count(distinct time) = 2 -- only observations before and after
               )
group by time
"""
control_results = pd.io.sql.read_sql(query, connection)

have the answer: did the hypothesis turn out to be true and the page does affect
the students’ behavior? - __YES__

Для таблицы test:


In [14]:
display(test_results)

,time,avg_diff
0,after,-104.6000
1,before,-60.5625


Абсолютное среднее значение поменялось на +72.6%, что сведетельствует о сильном изменении и влиянии сервиса.

Для категории control:

In [ ]:
display(control_results)

,time,avg_diff
0,after,-117.636364
1,before,-99.464286


Абсолютное среднее значение поменялось на +18.26% - несильно, чего и не следовало ожидать от категории студентов, которые не пользовались сервисом.

In [16]:
connection.close()